# Lab 0 bis — Going further
### Advanced Algorithms and Complexity — Master in AI

> 📌 This notebook **extends** Lab 0. It is meant for the end of the session if there is time left, or for self-study.

## 🎯 Objectives

**Part A — Advanced Git**
- work with **branches** (create, switch, merge, delete);
- understand and **resolve a merge conflict**;
- the **Pull Request** workflow on GitHub;
- cleanly **undo** a mistake (`restore`, `revert`, `stash`…).

**Part B — Advanced Python**
- iterators, generators, comprehensions;
- higher-order functions, **decorators**, **memoization**;
- the standard library tools that matter in algorithmics: `collections`, `heapq`, `bisect`, `itertools`, `timeit`;
- type hints, `dataclass`, exceptions, context managers;
- **NumPy**: vectorize instead of looping.

> 💡 Every topic is tied back to **complexity**, the common thread of this course.

---
# Part A — Advanced Git: branches

## A.1 Why branches?

A **branch** is an independent line of development. You can try an idea **without touching** the working version (`main`), then **merge** it if it is good, or throw it away if not.

```
main      ●───●───────────●   ← merge
               \         /
feature         ●───●───●     ← your experiments, isolated
```

Typical uses:
- try **another version of an algorithm** (e.g. iterative Fibonacci) without breaking the current one;
- **work as a team**: everyone on their own branch, then merge;
- one lab = one branch, merged into `main` when it is done.

A branch is just a **label** pointing to a commit: creating one is instant and copies no files.

## A.2 Essential commands

| Command | Effect |
|---|---|
| `git branch` | list branches (`*` = current branch) |
| `git switch -c tp1` | **create** branch `tp1` and move onto it |
| `git switch main` | go back to `main` |
| `git merge tp1` | merge `tp1` **into the current branch** |
| `git branch -d tp1` | delete `tp1` (once merged) |
| `git log --oneline --graph --all` | show the commit tree |
| `git push -u origin tp1` | publish the branch on GitHub |

> Older tutorials use `git checkout -b tp1`: it is equivalent to `git switch -c tp1`.

> In VS Code, the **branch name** is shown bottom left: click it to switch branches or create one.

## A.3 Live demo

The following cells create a **throwaway repository** in a temporary folder and play out a full scenario. Nothing in your real projects is touched.

The `git(...)` function runs a Git command and prints its output. It works on Windows, macOS and Linux.

In [ ]:
import subprocess, tempfile, pathlib

repo = pathlib.Path(tempfile.mkdtemp(prefix="demo_git_"))

def git(*args):
    # Runs `git <args>` in the demo repository and prints the output.
    r = subprocess.run(["git", "-c", "user.name=Demo", "-c", "user.email=demo@example.com", *args],
                       cwd=repo, capture_output=True, text=True)
    print(f"$ git {' '.join(args)}\n{r.stdout}{r.stderr}")

def write(name, text):
    (repo / name).write_text(text, encoding="utf-8")

git("init", "-b", "main")
write("fib.py", "def fib(n):\n    if n <= 1:\n        return n\n    return fib(n - 1) + fib(n - 2)\n")
git("add", "fib.py")
git("commit", "-m", "Naive recursive Fibonacci")

We create an `iterative` branch to try a faster version, **without touching** `main`:

In [ ]:
git("switch", "-c", "iterative")
write("fib.py", "def fib(n):\n    a, b = 0, 1\n    for _ in range(n):\n        a, b = b, a + b\n    return a\n")
git("commit", "-am", "Iterative Fibonacci")
git("switch", "main")
print("Content of fib.py on main:\n" + (repo / "fib.py").read_text())

👆 On `main`, the file has **not changed**: the work lives in the branch. Now we merge:

In [ ]:
git("merge", "iterative")
git("log", "--oneline", "--graph", "--all")
git("branch", "-d", "iterative")

Here `main` had not moved since the branch was created, so Git just **moved the label forward** (*fast-forward*).

## A.4 Merge conflicts

A **conflict** happens when **two branches change the same lines** differently. Git cannot guess which one to keep, so it asks you.

In [ ]:
git("switch", "-c", "author-ali")
write("README.md", "# Fibonacci lab\nAuthor: Ali\n")
git("add", "README.md"); git("commit", "-m", "README by Ali")

git("switch", "main")
write("README.md", "# Fibonacci lab\nAuthor: Sara\n")
git("add", "README.md"); git("commit", "-m", "README by Sara")

git("merge", "author-ali")
print((repo / "README.md").read_text())

Git wrote **both versions** into the file, between markers:

```
<<<<<<< HEAD          ← your version (current branch)
=======
>>>>>>> author-ali    ← the incoming version
```

**To resolve**:
1. edit the file to keep the right content and **remove the markers**;
2. `git add` the file;
3. `git commit` to finish the merge.

> In VS Code, buttons appear above the conflict: **Accept Current**, **Accept Incoming**, **Accept Both**. **Resolve in Merge Editor** opens a 3-pane view.

> To give up on a merge that went wrong: `git merge --abort`.

In [ ]:
write("README.md", "# Fibonacci lab\nAuthors: Ali and Sara\n")   # manual resolution
git("add", "README.md")
git("commit", "-m", "Merge: shared README")
git("log", "--oneline", "--graph", "--all")

This time the graph shows a real **merge commit**, with two parents.

## A.5 The Pull Request (PR) workflow on GitHub

In a team, you don't merge straight into `main`: you **propose** the merge through a *Pull Request*, which others review.

```bash
git switch -c tp1-loops              # 1. new branch
# ... work, git add, git commit ...
git push -u origin tp1-loops         # 2. publish the branch
```

3. On GitHub, a **Compare & pull request** banner appears: describe what changed and click **Create pull request**.
4. Teammates **review** (*Files changed* tab, line-by-line comments).
5. **Merge pull request** → the branch is merged into `main` on GitHub.
6. Locally, fetch the result and clean up:

```bash
git switch main
git pull
git branch -d tp1-loops
```

> **Fork**: to contribute to a repository you don't have rights on, you *fork* it (copy it into your account), work on it, then open a PR to the original.

## A.6 Undoing mistakes

| Situation | Command |
|---|---|
| I edited a file and want the last committed version back | `git restore file.py` |
| I ran `git add` by mistake | `git restore --staged file.py` |
| The message of my **last** commit is wrong (not pushed yet) | `git commit --amend -m "right message"` |
| Undo a commit that is **already pushed** | `git revert <id>` (creates an inverse commit: safe) |
| Go back and erase local commits | `git reset --hard <id>` ⚠️ **destructive** |
| Put my work aside to switch branches | `git stash` … then `git stash pop` |
| See what changed | `git diff` (unstaged), `git diff --staged` (staged) |

> ⚠️ Golden rule: **never rewrite history that is already pushed** (`reset`, `amend`, `rebase`) on a shared branch. Use `revert`.

In [ ]:
# Demo: stash and revert
write("fib.py", "# work in progress, unfinished\n")
git("status", "--short")
git("stash")                       # work is put aside
git("status", "--short")           # clean folder
git("stash", "pop")                # ... and it comes back
git("restore", "fib.py")           # in the end we drop it

write("notes.txt", "naive fib is fast\n")         # a mistake...
git("add", "notes.txt"); git("commit", "-m", "Note (wrong)")
git("revert", "--no-edit", "HEAD")  # cleanly undoes the last commit
git("log", "--oneline", "-4")

## A.7 Merge or rebase?

`git rebase main` (from your branch) **replays** your commits on top of `main`: history stays linear, with no merge commit.

```
before:  main ●───●───●          after rebase:   main ●───●───●
               \                                              \
         tp1    ●───●                                   tp1    ●'───●'
```

- **merge**: simple, rewrites nothing, the right default when starting out.
- **rebase**: cleaner history, but **rewrites** commits ⇒ only on a **personal, unshared** branch.

## A.8 Notebooks and Git: tips

- Outputs (images, tables) in `.ipynb` files bloat diffs. Before committing: *Clear All Outputs*, or install `nbstripout` (`pip install nbstripout && nbstripout --install`).
- A **tag** marks a specific version, ideal for a submission:

```bash
git tag tp1-submitted
git push origin tp1-submitted
```

## A.9 Suggested lab workflow

```
main ●───────────●───────────●
      \         / \         /
 tp1   ●──●──●─●   \       /
                tp2 ●──●──●
```

1. `git switch -c tp1` at the start of each lab;
2. regular commits during the session;
3. `git push -u origin tp1`, then a Pull Request → `main`;
4. a **tag** `tpN-submitted` on `main` marks the submitted version.

In [ ]:
import shutil
shutil.rmtree(repo, ignore_errors=True)    # removes the demo repository
print("Demo repository removed.")

---
# Part B — Advanced Python

## B.1 Comprehensions and generators

Comprehensions also exist for **dicts** and **sets**. With **parentheses**, you get a **generator**: values are produced **on demand**, never stored.

> 🔗 **Space complexity**: a list of $n$ elements uses memory proportional to $n$; a generator uses **constant** memory.

In [ ]:
import sys

squares_dict = {i: i * i for i in range(5)}
remainders   = {i % 3 for i in range(10)}
print(squares_dict, remainders)

n = 1_000_000
as_list = [i * i for i in range(n)]
as_gen  = (i * i for i in range(n))
print(f"list     : {sys.getsizeof(as_list):>10,} bytes")
print(f"generator: {sys.getsizeof(as_gen):>10,} bytes")
print("same sum:", sum(as_list) == sum(as_gen))

### Writing your own generator with `yield`

`yield` returns a value **and pauses the function** until the next value is requested.

In [ ]:
def fibonacci():
    a, b = 0, 1
    while True:            # infinite, but nothing is computed in advance
        yield a
        a, b = b, a + b

from itertools import islice
print(list(islice(fibonacci(), 15)))

## B.2 Unpacking, `enumerate`, `zip`, `*args`, `**kwargs`

In [ ]:
first, *middle, last = [1, 2, 3, 4, 5]
print(first, middle, last)

a, b = 1, 2
a, b = b, a                       # swap without a temporary variable
print(a, b)

names = ["Ali", "Sara", "Yacine"]
grades = [15, 17, 12]
for i, (name, grade) in enumerate(zip(names, grades), start=1):
    print(i, name, grade)

def summary(*values, unit="s", **options):
    print("values:", values, "| unit:", unit, "| options:", options)

summary(0.1, 0.2, 0.4, unit="ms", machine="LAB-03")

## B.3 Higher-order functions and `lambda`

In Python, a function is a **value**: you can pass it as an argument. This is used a lot to **sort by a criterion**.

In [ ]:
students = [("Ali", 15), ("Sara", 17), ("Yacine", 12), ("Lina", 17)]

print(sorted(students, key=lambda s: s[1], reverse=True))        # by grade, descending
print(sorted(students, key=lambda s: (-s[1], s[0])))             # grade ↓ then name ↑
print(max(students, key=lambda s: s[1]))
print(list(map(lambda x: x ** 2, range(6))))
print(list(filter(lambda x: x % 3 == 0, range(20))))

## B.4 Decorators: instrumenting an algorithm without changing it

A **decorator** is a function that takes a function and returns an **enhanced** version of it. Ideal for counting calls or timing **without touching the algorithm's code**.

> 🔗 In Lab 1 you had to add a `counter` by hand inside Fibonacci. With a decorator, you add it from the outside.

In [ ]:
import time
from functools import wraps

def count_calls(f):
    @wraps(f)                       # keeps f's name and docstring
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return f(*args, **kwargs)
    wrapper.calls = 0
    return wrapper

def timed(f):
    @wraps(f)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = f(*args, **kwargs)
        print(f"{f.__name__}{args}: {time.perf_counter() - start:.4f} s")
        return result
    return wrapper

@count_calls
def fib(n):
    return n if n <= 1 else fib(n - 1) + fib(n - 2)

for n in [10, 15, 20, 25]:
    fib.calls = 0
    fib(n)
    print(f"fib({n}) → {fib.calls:,} calls")

These are exactly the call counts from Lab 1 (177 for $n=10$). The `@timed` decorator is used the same way.

## B.5 Memoization: `functools.lru_cache`

Naive Fibonacci recomputes **the same values** millions of times. **Memoization** keeps each result in memory, so each `fib(k)` is computed **only once**.

> 🔗 This is the core idea of **dynamic programming** (chapter 4): trading **memory** for **time**.

In [ ]:
from functools import lru_cache

@timed
def fib_naive_timed(n):
    return fib(n)

@lru_cache(maxsize=None)
def fib_memo(n):
    return n if n <= 1 else fib_memo(n - 1) + fib_memo(n - 2)

fib_naive_timed(27)
start = time.perf_counter()
print("fib_memo(27)  =", fib_memo(27), f"in {time.perf_counter() - start:.6f} s")
print("fib_memo(300) =", fib_memo(300))
print(fib_memo.cache_info())

> ⚠️ Recursion in Python is limited (≈ 1000 nested calls by default, see `sys.getrecursionlimit()`). For large $n$, prefer an **iterative** version.

## B.6 `collections`: the right data structures

| Structure | Use it for |
|---|---|
| `Counter` | counting occurrences |
| `defaultdict` | dict with a default value (graphs, grouping) |
| `deque` | double-ended queue: add/remove **in constant time** at both ends |
| `namedtuple` | tuple with named fields |

In [ ]:
from collections import Counter, defaultdict, deque, namedtuple

text = "the cat and the dog and the rabbit"
print(Counter(text.split()).most_common(2))

graph = defaultdict(list)                   # adjacency list
for u, v in [("A", "B"), ("A", "C"), ("B", "D")]:
    graph[u].append(v)
print(dict(graph))

Point = namedtuple("Point", "x y")
p = Point(3, 4)
print(p.x, p.y, p)

### Experiment: `list.pop(0)` vs `deque.popleft()`

Removing the **first** element of a list forces Python to **shift all the others**. A `deque` does not have this problem.

In [ ]:
def drain_list(n):
    L = list(range(n))
    while L:
        L.pop(0)

def drain_deque(n):
    D = deque(range(n))
    while D:
        D.popleft()

print(f"{'n':>8} | {'list.pop(0)':>12} | {'deque.popleft':>13}")
for n in [5_000, 20_000, 80_000]:
    t1 = time.perf_counter(); drain_list(n); t1 = time.perf_counter() - t1
    t2 = time.perf_counter(); drain_deque(n); t2 = time.perf_counter() - t2
    print(f"{n:>8,} | {t1:>10.4f} s | {t2:>11.4f} s")

> 🧠 When $n$ is multiplied by 4, how does each column change?

## B.7 `heapq` and `bisect`

- `heapq`: a **heap** (priority queue). Instant access to the minimum, fast insert/remove. Used by Dijkstra, k-smallest, etc.
- `bisect`: **binary search** in a **sorted** list.

In [ ]:
import heapq, bisect, random

random.seed(1)
data = [random.randint(1, 1000) for _ in range(20)]
print("3 smallest:", heapq.nsmallest(3, data))

heap = []
for task, priority in [("submit lab", 1), ("read notes", 3), ("revise", 2)]:
    heapq.heappush(heap, (priority, task))
while heap:
    print(heapq.heappop(heap))

T = sorted(data)
x = T[7]
i = bisect.bisect_left(T, x)
print(f"{x} found at index {i}:", T[i] == x)

## B.8 `itertools` and combinatorial explosion

`itertools` generates permutations, combinations and Cartesian products. Their **count** explodes with $n$: that is why an algorithm that tries **every** possibility (brute force) quickly becomes unusable.

In [ ]:
from itertools import permutations, combinations, product
import math

print(list(permutations("ABC")))
print(list(combinations([1, 2, 3, 4], 2)))
print(list(product([0, 1], repeat=3)))

print(f"\n{'n':>3} | {'2^n':>15} | {'n!':>25}")
for n in [5, 10, 15, 20, 25]:
    print(f"{n:>3} | {2**n:>15,} | {math.factorial(n):>25,}")

> 🧠 At one billion tries per second, how long does it take to test all $25!$ visiting orders of 25 cities?

## B.9 Type hints and `dataclass`

**Type hints** document what a function expects and returns. Python does not check them at runtime, but VS Code (Pylance) and `mypy` use them to flag errors.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Measurement:
    algo: str
    n: int
    operations: int
    time: float
    notes: list[str] = field(default_factory=list)

    def ops_per_element(self) -> float:
        return self.operations / self.n

def fastest(results: list[Measurement]) -> Measurement:
    return min(results, key=lambda m: m.time)

results = [Measurement("traversal", 1000, 1000, 0.00004),
           Measurement("two_loops", 1000, 1_000_000, 0.045)]
print(results[1])
print(fastest(results).algo, results[1].ops_per_element())

## B.10 Exceptions and context managers

- `try / except`: catch an expected error instead of crashing.
- `raise`: signal invalid input.
- `with`: guarantees that some "cleanup" code always runs (closing a file, stopping a timer…).

In [ ]:
from contextlib import contextmanager

def int_sqrt(n: int) -> int:
    if n < 0:
        raise ValueError(f"n must be non-negative, got {n}")
    return math.isqrt(n)

for v in [16, -4]:
    try:
        print(int_sqrt(v))
    except ValueError as e:
        print("Error:", e)

@contextmanager
def stopwatch(label):
    start = time.perf_counter()
    try:
        yield
    finally:
        print(f"[{label}] {time.perf_counter() - start:.4f} s")

with stopwatch("sorting 1e6 elements"):
    sorted(random.random() for _ in range(1_000_000))

## B.11 Measuring properly with `timeit`

`timeit` repeats the measurement automatically and disables the garbage collector while timing: more reliable than a single `perf_counter`. In a notebook, the `%timeit` magic does the same in one line.

In [ ]:
import timeit

T = list(range(10_000))
S = set(T)
x = 9_999                                    # worst case: at the end

t_list = min(timeit.repeat(lambda: x in T, number=1000, repeat=5)) / 1000
t_set  = min(timeit.repeat(lambda: x in S, number=1000, repeat=5)) / 1000
print(f"x in list : {t_list * 1e6:8.2f} µs")
print(f"x in set  : {t_set * 1e6:8.2f} µs   ({t_list / t_set:,.0f}× faster)")

In [ ]:
%timeit sum(range(10_000))

## B.12 NumPy: vectorize instead of looping

NumPy runs operations on whole arrays in **compiled C code**. The number of operations stays the same, but each one costs **far less**.

> 🔗 Complexity describes **how** the cost grows with $n$; the constant in front (pure Python vs NumPy) changes, the **shape of the curve** does not.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sum_squares_python(L):
    s = 0
    for v in L:
        s += v * v
    return s

sizes, t_py, t_np = [10**k for k in range(2, 7)], [], []
for n in sizes:
    L = list(range(n)); A = np.arange(n, dtype=np.int64)
    t_py.append(min(timeit.repeat(lambda: sum_squares_python(L), number=1, repeat=3)))
    t_np.append(min(timeit.repeat(lambda: int((A * A).sum()), number=1, repeat=3)))

plt.loglog(sizes, t_py, "o-", label="Python loop")
plt.loglog(sizes, t_np, "s-", label="vectorized NumPy")
plt.xlabel("n"); plt.ylabel("time (s)"); plt.title("Same work, different constant")
plt.grid(True, which="both", alpha=0.3); plt.legend(); plt.show()

> 🧠 Are the two lines **parallel**? What does the vertical gap between them mean?

---
# Part C — Advanced exercises

In [ ]:
def check(label, got, expected):
    print(("✅" if got == expected else "❌"), label, "| got:", got, "| expected:", expected)

### Exercise A — Git: your first branch (in your real `aac-tp` repository)
1. Create a branch `tp0-bis`.
2. Add this notebook to it and make **at least 2 commits**.
3. Publish the branch (`git push -u origin tp0-bis`) and open a **Pull Request** into `main`.
4. Merge it on GitHub, then `git switch main && git pull` locally.
5. **Bonus**: deliberately create a conflict on `README.md` between two branches and resolve it in VS Code.

### Exercise B — Hand-made memoization
Write a decorator `memoize(f)` that stores results in a **dictionary**, **without** using `lru_cache`.

In [ ]:
def memoize(f):
    # TO DO (remove the raise line)
    raise NotImplementedError("memoize is not written yet")

try:
    @memoize
    def fib_m(n):
        return n if n <= 1 else fib_m(n - 1) + fib_m(n - 2)
    check("fib_m(80)", fib_m(80), 23416728348467685)   # without memoization this never finishes
except NotImplementedError as e:
    print("❌", e)

> ⚠️ If your `memoize` does not actually cache, the cell **never finishes** (≈ $10^{16}$ calls). Stop it with the ■ button.

### Exercise C — Binary search with a counter
Write `binary_search(T, x)` returning `(index, comparisons)` in a **sorted** list (`-1` if absent). Then compare the number of comparisons with $\lfloor \log_2 n \rfloor + 1$.

In [ ]:
def binary_search(T, x):
    # TO DO
    return -1, 0

T = list(range(0, 2000, 2))      # 1000 sorted even numbers
check("index of 1234", binary_search(T, 1234)[0], 617)
check("1235 absent", binary_search(T, 1235)[0], -1)
print("comparisons:", binary_search(T, 1234)[1], "| bound:", math.floor(math.log2(len(T))) + 1)

> ⚠️ For the counter to be comparable with $\lfloor \log_2 n \rfloor + 1$, count **one comparison per loop iteration**.

### Exercise D — The k most frequent words
Write `top_k(text, k)` returning the `k` most frequent words, from most to least frequent (use `Counter`).

In [ ]:
def top_k(text, k):
    # TO DO
    pass

check("top_k", top_k("a b a c b a d", 2), ["a", "b"])

### Exercise E — Prime number generator
Write a **generator** `primes()` that yields prime numbers one by one, forever.

In [ ]:
def primes():
    # TO DO
    yield from ()

check("first 10 primes", list(islice(primes(), 10)), [2, 3, 5, 7, 11, 13, 17, 19, 23, 29])

### Exercise F — Challenge: list vs set ⭐
For $n \in \{10^2, 10^3, 10^4, 10^5\}$, use `timeit` to measure `x in a_list` and `x in a_set` (worst case: `x` absent). Plot both curves on a **log-log** chart and describe the difference in **shape** (not just height).

In [ ]:
# TO DO

> *Your observations:* …

---
# 📚 Summary

| Topic | Key points |
|---|---|
| **Branches** | `switch -c`, `merge`, `branch -d`, one lab = one branch |
| **Conflicts** | `<<<<<<<` / `>>>>>>>` markers, edit → `add` → `commit` |
| **Pull Request** | `push -u`, PR on GitHub, review, merge, `pull` |
| **Undo** | `restore`, `revert` (safe), `stash`, never rewrite shared history |
| **Generators** | constant memory instead of proportional to $n$ |
| **Decorators** | instrument (count, time) without changing the algorithm |
| **Memoization** | `lru_cache`: memory for time → dynamic programming |
| **Structures** | `deque`, `Counter`, `defaultdict`, `heapq`, `bisect`: the right one changes the complexity |
| **Measuring** | `timeit`, `%timeit`; NumPy changes the constant, not the shape |

### Resources
- Pro Git (free book): https://git-scm.com/book/en/v2
- Learn branching interactively: https://learngitbranching.js.org/
- Python standard library: https://docs.python.org/3/library/
- NumPy for absolute beginners: https://numpy.org/doc/stable/user/absolute_beginners.html